# **NewsGenie — AI-Powered Information and News Assistant**
### Course End Project (CEP) — Applied Generative AI Specialisation

---

**Overview:**
NewsGenie is a unified, agentic AI platform that:
- 💬 **Handles conversations** — interprets and answers general queries via LLM
- 📰 **Fetches real-time news** — curates top stories by category (technology, finance, sports, health, science)
- 🔍 **Performs web searches** — enriches responses with live external information
- 🔀 **Routes intelligently** — LangGraph workflow classifies every query and dispatches it to the right handler
- 🛡️ **Handles failures gracefully** — fallback mechanisms for API errors, empty results, missing keys

**Architecture:** LangGraph StateGraph → Router → News/Search/Chat nodes → Streamlit UI


---
## Part 1: NewsGenie System Design
---

### **Setup: Environment & Imports**
> `OPENAI_API_KEY` required in `.env`. `NEWSAPI_KEY` optional — DuckDuckGo is used as fallback.

In [ ]:
import os, warnings, json, textwrap
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY", "")  # optional
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("NewsAPI key loaded:", bool(NEWSAPI_KEY))
print("(DuckDuckGo is used as no-key fallback for news and search)")


In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import requests
from datetime import datetime
from typing import TypedDict, List, Optional, Annotated
import operator

# ── LangGraph ─────────────────────────────────────────────────────────────────
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

# ── News / Search ─────────────────────────────────────────────────────────────
from duckduckgo_search import DDGS

# ── UI / Visualisation ────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

print("All imports successful.")


---
### **Step 1: LangGraph State Definition & Query Router**

Define the shared `NewsGenieState` that flows through the graph,
and a **Router node** that classifies every query as:
- `news` — requests for top headlines or category news
- `search` — requests needing live web information
- `chat` — general conversational queries


In [ ]:
class NewsGenieState(TypedDict):
    """Shared state flowing through the LangGraph workflow."""
    user_query    : str
    query_type    : str          # "news" | "search" | "chat"
    news_category : str          # technology | finance | sports | health | science | general
    news_results  : List[dict]   # fetched news articles
    search_results: str          # web search text
    chat_response : str          # LLM chat answer
    final_response: str          # assembled final answer
    error         : str          # error message if any
    messages      : Annotated[List, operator.add]  # conversation history

NEWS_CATEGORIES = ["technology", "finance", "sports", "health", "science", "general"]

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("NewsGenieState defined.")
print("Supported categories:", NEWS_CATEGORIES)


In [ ]:
def route_query(state: NewsGenieState) -> NewsGenieState:
    """
    Node 1 — Router
    Classifies the user query into: news | search | chat
    Also extracts the news category if applicable.
    """
    query = state["user_query"]

    router_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a query classifier for a news assistant.
Classify the user query into exactly ONE of these types:
- "news": user wants headlines, top stories, latest news, or news about a topic/category
- "search": user wants current/real-time facts, prices, scores, live data, or web info
- "chat": general question, opinion, explanation, or conversation

Also identify the news category if type is "news":
One of: technology, finance, sports, health, science, general

Respond in JSON only: {{"type": "...", "category": "..."}}
Examples:
- "What is the latest in AI?" → {{"type":"news","category":"technology"}}
- "What is machine learning?" → {{"type":"chat","category":"general"}}
- "Current Bitcoin price?" → {{"type":"search","category":"finance"}}
- "Top sports stories today?" → {{"type":"news","category":"sports"}}
"""),
        ("human", "{query}")
    ])

    try:
        result   = (router_prompt | llm).invoke({"query": query})
        content  = result.content.strip()
        # Extract JSON even if wrapped in markdown code block
        if "```" in content:
            content = content.split("```")[1].replace("json","").strip()
        parsed       = json.loads(content)
        query_type   = parsed.get("type", "chat")
        news_category = parsed.get("category", "general").lower()
        # Validate
        if query_type not in ("news", "search", "chat"):
            query_type = "chat"
        if news_category not in NEWS_CATEGORIES:
            news_category = "general"
    except Exception as e:
        query_type    = "chat"
        news_category = "general"

    print(f"  [Router] '{query[:60]}...' → type={query_type}, category={news_category}")
    return {**state, "query_type": query_type, "news_category": news_category, "error": ""}

print("route_query node defined.")


---
### **Step 2: News Fetching Node**

Fetches real-time news using:
1. **NewsAPI.org** (if `NEWSAPI_KEY` is set in `.env`)
2. **DuckDuckGo News** — automatic fallback (no API key required)

Returns up to 5 structured articles per query.


In [ ]:
def fetch_news_newsapi(category: str, query: str = "", max_results: int = 5) -> List[dict]:
    """Fetch news from NewsAPI.org (requires NEWSAPI_KEY)."""
    if not NEWSAPI_KEY:
        return []
    try:
        cat_map = {"finance": "business", "technology": "technology",
                   "sports": "sports", "health": "health",
                   "science": "science", "general": "general"}
        cat = cat_map.get(category, "general")
        params = {"apiKey": NEWSAPI_KEY, "language": "en",
                  "pageSize": max_results, "category": cat}
        if query:
            params["q"] = query
        resp = requests.get("https://newsapi.org/v2/top-headlines", params=params, timeout=8)
        if resp.status_code != 200:
            return []
        articles = resp.json().get("articles", [])
        return [{"title":   a.get("title",""),
                 "source":  a.get("source",{}).get("name",""),
                 "summary": a.get("description","") or "",
                 "url":     a.get("url",""),
                 "published": a.get("publishedAt","")[:10]} for a in articles if a.get("title")]
    except Exception:
        return []


def fetch_news_ddg(category: str, query: str = "", max_results: int = 5) -> List[dict]:
    """Fetch news via DuckDuckGo (no API key required — always available)."""
    search_term = query if query else f"{category} news today"
    try:
        with DDGS() as ddgs:
            raw = list(ddgs.news(search_term, max_results=max_results))
        return [{"title":    r.get("title",""),
                 "source":   r.get("source",""),
                 "summary":  r.get("body","")[:200],
                 "url":      r.get("url",""),
                 "published": r.get("date","")[:10]} for r in raw if r.get("title")]
    except Exception as e:
        return []


def fetch_news_node(state: NewsGenieState) -> NewsGenieState:
    """Node 2 — News Fetcher. Tries NewsAPI first, falls back to DuckDuckGo."""
    category = state.get("news_category", "general")
    query    = state["user_query"]

    # Try NewsAPI first
    articles = fetch_news_newsapi(category, query)
    source   = "NewsAPI"

    # Fallback to DuckDuckGo
    if not articles:
        articles = fetch_news_ddg(category, query)
        source   = "DuckDuckGo"

    if not articles:
        return {**state,
                "news_results":  [],
                "final_response": f"⚠️ No news found for '{category}'. Please try a different category or query.",
                "error": "no_news"}

    print(f"  [News] Fetched {len(articles)} articles via {source} for category='{category}'")
    return {**state, "news_results": articles, "error": ""}

print("fetch_news_node defined (NewsAPI + DuckDuckGo fallback).")


---
### **Step 3: Web Search Node**

For live data queries (prices, scores, current events),
DuckDuckGo text search provides up-to-date external information.


In [ ]:
def web_search_node(state: NewsGenieState) -> NewsGenieState:
    """Node 3 — Web Search via DuckDuckGo (no API key required)."""
    query = state["user_query"]
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=4))
        if not results:
            raise ValueError("No results")
        snippets = "\n\n".join(
            f"**{r.get('title','')}**\n{r.get('body','')[:300]}"
            for r in results
        )
        print(f"  [Search] Found {len(results)} results for: '{query[:60]}'")
        return {**state, "search_results": snippets, "error": ""}
    except Exception as e:
        # Fallback: ask LLM directly
        print(f"  [Search] DuckDuckGo failed ({e}), falling back to LLM knowledge")
        fallback = llm.invoke(f"Answer this based on your knowledge: {query}").content
        return {**state, "search_results": fallback, "error": "search_fallback"}

print("web_search_node defined.")


---
### **Step 4: Chat Response Node**

For general conversational queries, the LLM answers directly
using conversation history for context-aware, multi-turn dialogue.


In [ ]:
NEWSGENIE_SYSTEM = """You are NewsGenie, a helpful AI assistant specialising in
news, information, and general knowledge. You are friendly, concise, and accurate.
You help users stay informed about current events and answer their questions clearly.
When asked about news topics, suggest they select a category for the latest headlines."""

def chat_node(state: NewsGenieState) -> NewsGenieState:
    """Node 4 — Chat/LLM response for general queries."""
    query    = state["user_query"]
    history  = state.get("messages", [])

    messages = [SystemMessage(content=NEWSGENIE_SYSTEM)]
    # Include recent history (last 6 messages for context window efficiency)
    messages.extend(history[-6:])
    messages.append(HumanMessage(content=query))

    response = llm.invoke(messages)
    print(f"  [Chat] Answered: '{query[:60]}'")
    return {**state, "chat_response": response.content, "error": ""}

print("chat_node defined.")


---
### **Step 5: Response Formatter & LangGraph Workflow Assembly**

The **formatter node** assembles the final response from whichever handler ran.
The **StateGraph** wires all nodes together with conditional routing.


In [ ]:
def format_response_node(state: NewsGenieState) -> NewsGenieState:
    """Node 5 — Format the final response based on query type."""
    qtype = state.get("query_type", "chat")

    if state.get("error") == "no_news":
        return state  # already set final_response

    if qtype == "news":
        articles = state.get("news_results", [])
        category = state.get("news_category", "general").title()
        if not articles:
            final = f"No {category} news found right now. Try again shortly."
        else:
            lines = [f"📰 **Top {category} News** ({datetime.now().strftime('%b %d, %Y')})\n"]
            for i, a in enumerate(articles, 1):
                lines.append(f"{i}. **{a['title']}**")
                if a.get("summary"):
                    lines.append(f"   {a['summary'][:150]}...")
                if a.get("source"):
                    lines.append(f"   *Source: {a['source']}*")
                lines.append("")
            final = "\n".join(lines)

    elif qtype == "search":
        snippets = state.get("search_results", "")
        if snippets:
            # Let LLM synthesise the search results into a clean answer
            synth_prompt = f"""Using these search results, answer the user's question concisely.
Question: {state['user_query']}

Search Results:
{snippets[:2000]}

Provide a clear, direct answer in 2-4 sentences."""
            final = llm.invoke(synth_prompt).content
        else:
            final = "I couldn't find current information for that query. Please try rephrasing."

    else:  # chat
        final = state.get("chat_response", "I'm not sure how to answer that. Could you rephrase?")

    # Append to message history
    new_msgs = [
        HumanMessage(content=state["user_query"]),
        AIMessage(content=final)
    ]
    return {**state, "final_response": final, "messages": new_msgs}

print("format_response_node defined.")


In [ ]:
def decide_route(state: NewsGenieState) -> str:
    """Conditional edge: directs flow after router based on query_type."""
    return state.get("query_type", "chat")

# ── Build the StateGraph ──────────────────────────────────────────────────────
graph = StateGraph(NewsGenieState)

graph.add_node("router",     route_query)
graph.add_node("news",       fetch_news_node)
graph.add_node("search",     web_search_node)
graph.add_node("chat",       chat_node)
graph.add_node("formatter",  format_response_node)

graph.add_edge(START, "router")
graph.add_conditional_edges(
    "router",
    decide_route,
    {"news": "news", "search": "search", "chat": "chat"}
)
graph.add_edge("news",   "formatter")
graph.add_edge("search", "formatter")
graph.add_edge("chat",   "formatter")
graph.add_edge("formatter", END)

memory   = MemorySaver()
newsgenie = graph.compile(checkpointer=memory)

print("NewsGenie LangGraph workflow compiled.")
print("Nodes:", ["router","news","search","chat","formatter"])
print("Routing: START → router → {news|search|chat} → formatter → END")


In [ ]:
# ── Mermaid-style workflow diagram (text) ─────────────────────────────────────
print("""
NewsGenie Workflow:
==================
     [User Query]
          │
     [Router Node]          ← classifies: news | search | chat
    /       |       \
[News]  [Search]  [Chat]    ← specialised handlers
    \       |       /
     [Formatter Node]       ← assembles final response
          │
    [Final Answer]
          │
   [MemorySaver] ←── stores conversation history
""")


---
### **Step 6: Running NewsGenie — Sample Scenarios**


In [ ]:
def ask_newsgenie(query: str, session_id: str = "default", category_hint: str = "") -> dict:
    """Run a query through the NewsGenie workflow and return the result."""
    config = {"configurable": {"thread_id": session_id}}
    initial_state: NewsGenieState = {
        "user_query":     query,
        "query_type":     "",
        "news_category":  category_hint or "general",
        "news_results":   [],
        "search_results": "",
        "chat_response":  "",
        "final_response": "",
        "error":          "",
        "messages":       [],
    }
    result = newsgenie.invoke(initial_state, config=config)
    return result

def display_result(result: dict):
    """Pretty-print a NewsGenie result."""
    print(f"Query type : {result.get('query_type','?')}")
    print(f"Category   : {result.get('news_category','?')}")
    if result.get("error"):
        print(f"Error      : {result['error']}")
    print()
    print(result.get("final_response","(no response)"))
    print()


#### Scenario 1: Technology News

In [ ]:
print("=" * 65)
print("SCENARIO 1: Technology News")
print("=" * 65)
r1 = ask_newsgenie("What are the latest technology news stories today?", "s1")
display_result(r1)


#### Scenario 2: Finance News

In [ ]:
print("=" * 65)
print("SCENARIO 2: Finance / Business News")
print("=" * 65)
r2 = ask_newsgenie("Show me top finance and business news", "s2")
display_result(r2)


#### Scenario 3: Sports News

In [ ]:
print("=" * 65)
print("SCENARIO 3: Sports News")
print("=" * 65)
r3 = ask_newsgenie("What are the top sports headlines?", "s3")
display_result(r3)


#### Scenario 4: General Chat Query

In [ ]:
print("=" * 65)
print("SCENARIO 4: General Chat")
print("=" * 65)
r4 = ask_newsgenie("What is artificial intelligence and how does it work?", "s4")
display_result(r4)


#### Scenario 5: Live Web Search

In [ ]:
print("=" * 65)
print("SCENARIO 5: Live Web Search (Current Data)")
print("=" * 65)
r5 = ask_newsgenie("What is the latest news about OpenAI GPT models?", "s5")
display_result(r5)


#### Scenario 6: Multi-Turn Memory (Follow-up)

In [ ]:
print("=" * 65)
print("SCENARIO 6: Multi-Turn Memory")
print("=" * 65)
SESSION = "multi_turn"
r6a = ask_newsgenie("Tell me about health news today.", SESSION)
print("Turn 1:", r6a.get("final_response","")[:200], "...\n")

r6b = ask_newsgenie("Can you summarise what you just told me in one sentence?", SESSION)
print("Turn 2 (follow-up):")
display_result(r6b)


---
### **Step 7: Automated Test Case Execution**

Run all 10 test cases and record PASS/FAIL against expected criteria.


In [ ]:
import time

test_cases = [
    # (id, query, session, expected_type, expected_category, description)
    ("TC-01", "What are the latest technology news?",         "tc1", "news",   "technology", "Tech news routing"),
    ("TC-02", "Show me top finance headlines today",          "tc2", "news",   "finance",    "Finance news routing"),
    ("TC-03", "What are today's top sports stories?",         "tc3", "news",   "sports",     "Sports news routing"),
    ("TC-04", "What is machine learning?",                    "tc4", "chat",   "general",    "General chat query"),
    ("TC-05", "Explain the difference between AI and ML",     "tc5", "chat",   "general",    "Conceptual chat query"),
    ("TC-06", "Latest news in health and medicine",           "tc6", "news",   "health",     "Health news routing"),
    ("TC-07", "What's happening in science today?",           "tc7", "news",   "science",    "Science news routing"),
    ("TC-08", "Who is the CEO of OpenAI?",                    "tc8", "search", "general",    "Web search routing"),
    ("TC-09", "What is ChatGPT used for?",                    "tc9", "chat",   "general",    "Product info chat"),
    ("TC-10", "Give me the latest news about space missions", "tc10","news",   "science",    "Space/science news"),
]

results_log = []
print(f"Running {len(test_cases)} test cases...\n")

for tc_id, query, session, exp_type, exp_cat, desc in test_cases:
    try:
        result = ask_newsgenie(query, session)
        actual_type = result.get("query_type", "")
        actual_cat  = result.get("news_category", "")
        has_response = bool(result.get("final_response","").strip())

        type_pass = actual_type == exp_type
        resp_pass = has_response

        # For news, check we got articles or a valid response
        if exp_type == "news":
            articles  = result.get("news_results", [])
            data_pass = len(articles) > 0 or has_response
        else:
            data_pass = has_response

        status = "PASS" if (type_pass and resp_pass and data_pass) else "FAIL"
        results_log.append({
            "ID": tc_id, "Description": desc, "Query": query[:50]+"...",
            "Exp Type": exp_type, "Got Type": actual_type,
            "Exp Cat": exp_cat, "Got Cat": actual_cat,
            "Has Response": has_response, "Status": status
        })
        print(f"  [{status}] {tc_id}: {desc}")
    except Exception as e:
        results_log.append({"ID": tc_id, "Description": desc, "Query": query[:50]+"...",
                            "Status": "ERROR", "Got Type": str(e)[:40]})
        print(f"  [ERROR] {tc_id}: {e}")
    time.sleep(1)  # rate limit courtesy

print()
df_results = pd.DataFrame(results_log)
pass_count = (df_results["Status"] == "PASS").sum()
print(f"Results: {pass_count}/{len(test_cases)} PASSED ({pass_count/len(test_cases)*100:.0f}%)")


In [ ]:
# Display full test results table
print("\n=== Full Test Results ===")
display_cols = ["ID","Description","Exp Type","Got Type","Exp Cat","Got Cat","Status"]
print(df_results[display_cols].to_string(index=False))


---
### **Step 8: Visualisations — Workflow & Results**


In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi":120,"axes.titlesize":12})


In [ ]:
# ── Chart 1: Test Results Summary ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

status_counts = df_results["Status"].value_counts()
colors_s = {"PASS": "#2A9D8F", "FAIL": "#E76F51", "ERROR": "#E9C46A"}
bar_colors = [colors_s.get(s, "#ccc") for s in status_counts.index]
bars = axes[0].bar(status_counts.index, status_counts.values, color=bar_colors, width=0.4)
axes[0].bar_label(bars, padding=3, fontsize=11)
axes[0].set_title("Test Case Results", fontweight="bold")
axes[0].set_ylabel("Count"); axes[0].set_ylim(0, len(test_cases) + 2)

# Query type distribution
type_counts = df_results["Got Type"].value_counts()
axes[1].pie(type_counts.values, labels=type_counts.index,
            autopct="%1.0f%%", colors=["#2E86AB","#A23B72","#F18F01"],
            startangle=90)
axes[1].set_title("Query Type Distribution", fontweight="bold")

plt.suptitle("NewsGenie Test Summary", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz_01_test_results.png", bbox_inches="tight")
plt.show()
print("Saved viz_01_test_results.png")


In [ ]:
# ── Chart 2: News Articles fetched per category ───────────────────────────────
news_results = [r for r in results_log if r.get("Got Type") == "news"]
cat_counts   = {}
for r in results_log:
    cat = r.get("Got Cat","unknown")
    cat_counts[cat] = cat_counts.get(cat, 0) + 1

fig, ax = plt.subplots(figsize=(9, 4))
colors_c = ["#264653","#2A9D8F","#E9C46A","#F18F01","#E76F51","#2E86AB"]
cats = list(cat_counts.keys())
vals = [cat_counts[c] for c in cats]
bars = ax.bar([c.title() for c in cats], vals, color=colors_c[:len(cats)])
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title("Queries Processed per Category", fontweight="bold")
ax.set_ylabel("Count"); ax.set_xlabel("Category")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("viz_02_category_distribution.png", bbox_inches="tight")
plt.show()
print("Saved viz_02_category_distribution.png")


In [ ]:
# ── Chart 3: LangGraph Workflow Diagram ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")

def box(ax, x, y, w, h, text, color):
    rect = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
           boxstyle="round,pad=0.1", facecolor=color, edgecolor="#333", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=10, fontweight="bold", color="white")

def arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->", color="#555", lw=1.5))

# Nodes
box(ax, 5, 5.2, 2.5, 0.7, "User Query", "#2E86AB")
box(ax, 5, 4.0, 2.5, 0.7, "Router Node
(classify intent)", "#A23B72")
box(ax, 2, 2.5, 2.2, 0.7, "News Node
(DuckDuckGo/NewsAPI)", "#2A9D8F")
box(ax, 5, 2.5, 2.2, 0.7, "Search Node
(DuckDuckGo Web)", "#F18F01")
box(ax, 8, 2.5, 2.2, 0.7, "Chat Node
(GPT-4o-mini)", "#264653")
box(ax, 5, 1.0, 2.5, 0.7, "Formatter Node
(assemble response)", "#E76F51")

# Arrows
arrow(ax, 5, 4.85, 5, 4.35)
arrow(ax, 3.7, 3.65, 2.5, 2.85)
arrow(ax, 5,   3.65, 5,   2.85)
arrow(ax, 6.3, 3.65, 7.5, 2.85)
arrow(ax, 2, 2.15, 4.0, 1.35)
arrow(ax, 5, 2.15, 5,   1.35)
arrow(ax, 8, 2.15, 6.0, 1.35)

# Labels
ax.text(3.2, 3.35, "news",   fontsize=8, color="#2A9D8F", style="italic")
ax.text(5.1, 3.35, "search", fontsize=8, color="#F18F01", style="italic")
ax.text(6.5, 3.35, "chat",   fontsize=8, color="#264653", style="italic")

ax.text(5, 0.35, "MemorySaver (conversation history)", ha="center",
        fontsize=9, color="#555", style="italic")

ax.set_title("NewsGenie — LangGraph Workflow Architecture", fontsize=13,
             fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("viz_03_workflow_diagram.png", bbox_inches="tight")
plt.show()
print("Saved viz_03_workflow_diagram.png")


---
## Conclusion

**NewsGenie** successfully demonstrates a production-grade agentic news assistant:

| Component | Implementation |
|---|---|
| Query Classification | LangGraph Router Node — LLM-powered intent detection |
| News Fetching | NewsAPI.org (primary) + DuckDuckGo News (fallback) |
| Web Search | DuckDuckGo text search + LLM synthesis |
| Chat | GPT-4o-mini with conversation history |
| Workflow | LangGraph `StateGraph` — 5 nodes, conditional routing |
| Memory | `MemorySaver` — multi-turn session context |
| Error Handling | API fallbacks, empty result handling, LLM fallback |
| Evaluation | 10 automated test cases — routing + response validation |
| UI | Streamlit dashboard with category selector and chat |

**Capabilities demonstrated:**
- ✅ Technology, finance, sports, health, science news categories
- ✅ Query type routing (news vs search vs chat)
- ✅ Graceful fallback when NewsAPI is unavailable
- ✅ Multi-turn conversational memory
- ✅ Live web search synthesis via LLM
---
